In [1]:
import sys
sys.path.append("..")

import pandas as pd
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import StratifiedKFold, cross_validate
from sklearn.pipeline import Pipeline

from src.data_prep import load_data, split_data, build_preprocessor

df = load_data()
X_train, X_test, y_train, y_test = split_data(df)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
scoring = ["precision", "recall", "f1", "roc_auc"]


def cv_scores(model):
    scores = cross_validate(model, X_train, y_train, cv=cv, scoring=scoring)
    return {m: round(scores[f"test_{m}"].mean(), 3) for m in scoring}


plain = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", LogisticRegression(max_iter=1000, random_state=42)),
])

balanced = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", LogisticRegression(max_iter=1000, random_state=42, class_weight="balanced")),
])

pd.DataFrame({
    "LogReg (plain)": cv_scores(plain),
    "LogReg (balanced)": cv_scores(balanced),
}).T

,precision,recall,f1,roc_auc
LogReg (plain),0.654,0.544,0.593,0.846
LogReg (balanced),0.517,0.801,0.628,0.846


In [2]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.model_selection import GridSearchCV

rf = Pipeline([
    ("preprocessor", build_preprocessor()),
    ("model", RandomForestClassifier(
        n_estimators=200, class_weight="balanced", random_state=42
    )),
])

param_grid = {
    "model__max_depth": [4, 6, 8, 12],
    "model__min_samples_leaf": [5, 10, 20],
}

grid = GridSearchCV(rf, param_grid, cv=cv, scoring="roc_auc", n_jobs=-1)
grid.fit(X_train, y_train)

print(grid.best_params_)
print(round(grid.best_score_, 3))

{'model__max_depth': 8, 'model__min_samples_leaf': 10}
0.848


In [3]:
from sklearn.model_selection import cross_val_predict
from sklearn.metrics import precision_score, recall_score, f1_score

candidates = {
    "LogReg (plain)": plain,
    "Random Forest (tuned)": grid.best_estimator_,
}

oof = {}
for name, model in candidates.items():
    oof[name] = cross_val_predict(
        model, X_train, y_train, cv=cv, method="predict_proba"
    )[:, 1]

rows = []
for name, proba in oof.items():
    for t in [0.2, 0.25, 0.3, 0.35, 0.4, 0.5]:
        pred = (proba >= t).astype(int)
        rows.append({
            "Model": name,
            "Threshold": t,
            "Precision": precision_score(y_train, pred),
            "Recall": recall_score(y_train, pred),
            "F1": f1_score(y_train, pred),
        })

pd.DataFrame(rows).round(3)

,Model,Threshold,Precision,Recall,F1
0,LogReg (plain),0.20,0.475,0.854,0.611
1,LogReg (plain),0.25,0.510,0.817,0.628
2,LogReg (plain),0.30,0.541,0.767,0.635
3,LogReg (plain),0.35,0.570,0.717,0.635
4,LogReg (plain),0.40,0.594,0.664,0.627
5,LogReg (plain),0.50,0.654,0.544,0.594
6,Random Forest (tuned),0.20,0.369,0.969,0.535
7,Random Forest (tuned),0.25,0.398,0.945,0.560
8,Random Forest (tuned),0.30,0.425,0.924,0.582
9,Random Forest (tuned),0.35,0.451,0.902,0.602


In [4]:
from sklearn.metrics import confusion_matrix, roc_auc_score

plain.fit(X_train, y_train)
proba = plain.predict_proba(X_test)[:, 1]

THRESHOLD = 0.30
rows = {}
for label, t in [("Baseline (threshold 0.50)", 0.50), ("Improved (threshold 0.30)", THRESHOLD)]:
    pred = (proba >= t).astype(int)
    rows[label] = {
        "Precision": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
        "ROC-AUC": roc_auc_score(y_test, proba),
    }
    print(label)
    print(confusion_matrix(y_test, pred))

pd.DataFrame(rows).T.round(3)

Baseline (threshold 0.50)
[[926 109]
 [165 209]]
Improved (threshold 0.30)
[[774 261]
 [ 92 282]]


,Precision,Recall,F1,ROC-AUC
Baseline (threshold 0.50),0.657,0.559,0.604,0.842
Improved (threshold 0.30),0.519,0.754,0.615,0.842
